# Airplane Boarding Strategy Simulation
## Analysis Notebook
**Course:** Modeling and Simulation (University Project)

This notebook loads the experiment results, runs statistical tests, and reproduces all six charts.

> **Run from the `boarding_sim/` directory** after executing `run_baseline.py` and `run_experiments.py`.


In [ ]:
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120
from IPython.display import display, Image

# Ensure the boarding package is importable
import sys
sys.path.insert(0, str(pathlib.Path(".").resolve()))

RES = pathlib.Path("results")
print("Results directory:", RES.resolve())
print("Files found:", [f.name for f in sorted(RES.glob("*.csv"))])


## 1. Load experiment results


In [ ]:
df_base = pd.read_csv(RES / "exp1_baseline.csv")
df_ph   = pd.read_csv(RES / "exp2_philippine.csv")
df_comp = pd.read_csv(RES / "exp3_compliance_sweep.csv")
df_grp  = pd.read_csv(RES / "exp4_group_sweep.csv")
df_sens = pd.read_csv(RES / "exp5_sensitivity.csv")
df_abl  = pd.read_csv(RES / "exp6_ablation.csv")

print(f"Exp 1 (baseline):    {len(df_base)} rows  ({df_base.rep_id.nunique()} reps x {df_base.strategy.nunique()} strategies)")
print(f"Exp 2 (Philippine):  {len(df_ph)} rows")
print(f"Exp 3 (compliance):  {len(df_comp)} rows  ({df_comp.compliance_rate.nunique()} levels)")
print(f"Exp 4 (group frac):  {len(df_grp)} rows  ({df_grp.group_fraction.nunique()} levels)")
print(f"Exp 5 (sensitivity): {len(df_sens)} rows  ({df_sens.stow_multiplier.nunique()} x {df_sens.walk_speed_mean.nunique()} conditions)")
print(f"Exp 6 (ablation):    {len(df_abl)} rows  ({df_abl.ablation_feature.nunique()} features)")


## 2. Descriptive statistics


In [ ]:
from boarding.stats import summary_table

print("=== Baseline ===")
display(summary_table(df_base))
print()
print("=== Philippine ===")
display(summary_table(df_ph))


## 3. Strategy ranking


In [ ]:
from boarding.stats import rank_strategies, one_way_anova

print("--- Baseline ranking ---")
display(rank_strategies(df_base))

print()
print("--- Philippine ranking ---")
display(rank_strategies(df_ph))

print()
print("--- ANOVA (baseline) ---")
anova = one_way_anova(df_base)
print(f"F = {anova['F']},  p = {anova['p']:.2e},  significant: {anova['significant_005']}")


## 4. Statistical tests

### 4a. Paired t-test (primary — CRN makes pairing valid)


In [ ]:
from boarding.stats import paired_ttest_pairwise

pt = paired_ttest_pairwise(df_base)
display(pt[["group_a","group_b","t_stat","p_bonferroni","cohens_d","ci_lo_95","ci_hi_95","significant_005"]])


### 4b. Mann-Whitney U (non-parametric check)


In [ ]:
from boarding.stats import mannwhitney_pairwise

mw = mannwhitney_pairwise(df_base)
display(mw[["group_a","group_b","U_stat","p_bonferroni","cohens_d","significant_005"]])


### 4c. Scenario comparison (Philippine vs Baseline)


In [ ]:
from boarding.stats import compare_scenarios

sc = compare_scenarios(df_base, df_ph)
display(sc)


## 5. Charts

### Fig 1 — Boxplot of total boarding time (baseline)


In [ ]:
display(Image(str(RES / "fig1_boxplot_baseline.png")))


### Fig 2 — Baseline vs Philippine (side by side)


In [ ]:
display(Image(str(RES / "fig2_boxplot_comparison.png")))


### Fig 1b — Interference events by strategy


In [ ]:
display(Image(str(RES / "fig1b_interference_baseline.png")))


### Fig 3 — Effect of compliance rate on boarding time


In [ ]:
display(Image(str(RES / "fig3_compliance_line.png")))


### Fig 4 — Effect of group travel fraction on boarding time


In [ ]:
display(Image(str(RES / "fig4_group_line.png")))


### Fig 5 — Sensitivity analysis (stow time × walk speed)


In [ ]:
display(Image(str(RES / "fig5_sensitivity_heatmap.png")))


### Fig 6 — Ablation: which behaviour feature contributes most to slowdown?


In [ ]:
display(Image(str(RES / "fig6_ablation_bars.png")))


## 6. Strategy ranking under each sensitivity condition


In [ ]:
from boarding.stats import rank_strategies

# Ranking changes as stow_multiplier increases (does Steffen stay fastest?)
rank_by_stow = rank_strategies(df_sens, condition_col="stow_multiplier")
print("Ranking by stow_multiplier (1 = fastest):")
display(rank_by_stow)

rank_by_walk = rank_strategies(df_sens, condition_col="walk_speed_mean")
print("\nRanking by walk_speed_mean (1 = fastest):")
display(rank_by_walk)


## 7. Compliance sweep — rank changes


In [ ]:
rank_by_comp = rank_strategies(df_comp, condition_col="compliance_rate")
print("Ranking by compliance_rate (1 = fastest; 1.0 = all compliant):")
display(rank_by_comp)


## 8. Summary and conclusions

Based on the simulation results:

1. **Steffen** is the fastest strategy under both the baseline and Philippine scenarios,
   consistently ranked #1 across all sensitivity conditions.

2. **Back-to-Front** is consistently the *slowest* strategy — slower even than Random,
   consistent with published findings (Steffen 2008, Milne & Kelly).

3. **Philippine behaviour adds significant delay** for all strategies:
   - Steffen: +76% (from 687 s to 1211 s)
   - Outside-In: +32%
   - Random: +18%
   - Back-to-Front: +9%

4. **Compliance rate** has the largest single impact: as compliance drops from 1.0 to 0.0,
   boarding time increases substantially for all strategies.

5. **Steffen remains robust** — it stays fastest even when stow time is doubled or
   walk speed is slowed. The strategy ranking is stable under sensitivity analysis.

> **All ASSUMED parameters are labeled in `config.yaml`. Do not cite these as literature values.**
